# Cryptocurrency Price Forecasting
## Data Preprocessing and Feature Engineering

### Objective

To transform the cleaned Bitcoin OHLCV dataset into a
machine-learning-ready dataset for next-day closing price
forecasting.

### Steps

1. Load cleaned Bitcoin data
2. Create financial and technical features
3. Create the next-day Close target
4. Check for data leakage
5. Split the data chronologically
6. Save Train, Validation and Test datasets

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from ta.momentum import RSIIndicator
from ta.trend import MACD

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Input and output paths

INPUT_PATH = Path(
    "../data/processed/bitcoin_clean.csv"
)

OUTPUT_PATH = Path(
    "../data/processed"
)

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Input file:",
    INPUT_PATH.resolve()
)

print(
    "Output folder:",
    OUTPUT_PATH.resolve()
)

Input file: C:\Crypto_Forecasting_Project\LSTM_Light_BGM\data\processed\bitcoin_clean.csv
Output folder: C:\Crypto_Forecasting_Project\LSTM_Light_BGM\data\processed


In [3]:
# Check whether the cleaned dataset exists

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"File not found:\n{INPUT_PATH.resolve()}\n\n"
        "Run 01_eda.ipynb first."
    )

df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["Date"]
)

print(
    "Dataset loaded successfully."
)

print(
    "Shape:",
    df.shape
)

display(
    df.head()
)

Dataset loaded successfully.
Shape: (3248, 6)


,Date,Open,High,Low,Close,Volume
0,2013-05-05,112.900002,118.800003,107.142998,115.910004,0.0
1,2013-05-06,115.980003,124.663002,106.639999,112.300003,0.0
2,2013-05-07,112.250000,113.444000,97.699997,111.500000,0.0
3,2013-05-08,109.599998,115.779999,109.599998,113.566002,0.0
4,2013-05-09,113.199997,113.459999,109.260002,112.669998,0.0


In [4]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate dates:")
print(df["Date"].duplicated().sum())

print("\nDates sorted:")
print(df["Date"].is_monotonic_increasing)

Columns:
['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

Missing values:


Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


Duplicate dates:
0

Dates sorted:
True


In [5]:
df = df.sort_values(
    "Date"
).reset_index(drop=True)

print(
    "Chronological ordering:",
    df["Date"].is_monotonic_increasing
)

Chronological ordering: True


<b>FORMULA:</b> Return(t) = Close(t) / Close(t-1) - 1

In [6]:
df["Return"] = (
    df["Close"].pct_change()
)

display(
    df[
        [
            "Date",
            "Close",
            "Return"
        ]
    ].head(10)
)

,Date,Close,Return
0,2013-05-05,115.910004,NaN
1,2013-05-06,112.300003,-0.031145
2,2013-05-07,111.500000,-0.007124
3,2013-05-08,113.566002,0.018529
4,2013-05-09,112.669998,-0.007890
5,2013-05-10,117.199997,0.040206
6,2013-05-11,115.242996,-0.016698
7,2013-05-12,115.000000,-0.002109
8,2013-05-13,117.980003,0.025913
9,2013-05-14,111.500000,-0.054925


SMA - Simple Moving Average

In [7]:
df["SMA_7"] = (
    df["Close"]
    .rolling(window=7)
    .mean()
)

display(
    df[
        [
            "Date",
            "Close",
            "SMA_7"
        ]
    ].head(15)
)

,Date,Close,SMA_7
0,2013-05-05,115.910004,NaN
1,2013-05-06,112.300003,NaN
2,2013-05-07,111.500000,NaN
3,2013-05-08,113.566002,NaN
4,2013-05-09,112.669998,NaN
5,2013-05-10,117.199997,NaN
6,2013-05-11,115.242996,114.055571
7,2013-05-12,115.000000,113.925571
8,2013-05-13,117.980003,114.737000
9,2013-05-14,111.500000,114.737000


In [8]:
df["SMA_14"] = (
    df["Close"]
    .rolling(window=14)
    .mean()
)

display(
    df[
        [
            "Date",
            "Close",
            "SMA_7",
            "SMA_14"
        ]
    ].head(20)
)

,Date,Close,SMA_7,SMA_14
0,2013-05-05,115.910004,NaN,NaN
1,2013-05-06,112.300003,NaN,NaN
2,2013-05-07,111.500000,NaN,NaN
3,2013-05-08,113.566002,NaN,NaN
4,2013-05-09,112.669998,NaN,NaN
5,2013-05-10,117.199997,NaN,NaN
6,2013-05-11,115.242996,114.055571,NaN
7,2013-05-12,115.000000,113.925571,NaN
8,2013-05-13,117.980003,114.737000,NaN
9,2013-05-14,111.500000,114.737000,NaN


In [9]:
df["Volatility_7"] = (
    df["Return"]
    .rolling(window=7)
    .std()
)

display(
    df[
        [
            "Date",
            "Return",
            "Volatility_7"
        ]
    ].head(20)
)

,Date,Return,Volatility_7
0,2013-05-05,NaN,NaN
1,2013-05-06,-0.031145,NaN
2,2013-05-07,-0.007124,NaN
3,2013-05-08,0.018529,NaN
4,2013-05-09,-0.007890,NaN
5,2013-05-10,0.040206,NaN
6,2013-05-11,-0.016698,NaN
7,2013-05-12,-0.002109,0.023525
8,2013-05-13,0.025913,0.021050
9,2013-05-14,-0.054925,0.031602


In [10]:
rsi_indicator = RSIIndicator(
    close=df["Close"],
    window=14
)

df["RSI_14"] = (
    rsi_indicator.rsi()
)

display(
    df[
        [
            "Date",
            "Close",
            "RSI_14"
        ]
    ].head(20)
)

,Date,Close,RSI_14
0,2013-05-05,115.910004,NaN
1,2013-05-06,112.300003,NaN
2,2013-05-07,111.500000,NaN
3,2013-05-08,113.566002,NaN
4,2013-05-09,112.669998,NaN
5,2013-05-10,117.199997,NaN
6,2013-05-11,115.242996,NaN
7,2013-05-12,115.000000,NaN
8,2013-05-13,117.980003,NaN
9,2013-05-14,111.500000,NaN


In [11]:
macd_indicator = MACD(
    close=df["Close"],
    window_slow=26,
    window_fast=12,
    window_sign=9
)

df["MACD"] = (
    macd_indicator.macd()
)

df["MACD_Signal"] = (
    macd_indicator.macd_signal()
)

display(
    df[
        [
            "Date",
            "Close",
            "MACD",
            "MACD_Signal"
        ]
    ].tail(10)
)

,Date,Close,MACD,MACD_Signal
3238,2022-09-05,19812.371479,-1658.960134,-1780.499896
3239,2022-09-11,21769.256340,-1500.476142,-1724.495145
3240,2022-09-12,22370.448650,-1311.249969,-1641.846110
3241,2022-09-18,19419.506191,-1383.455642,-1590.168016
3242,2022-09-19,19544.129250,-1414.319721,-1554.998357
3243,2022-09-25,18802.097976,-1481.576754,-1540.314037
3244,2022-10-02,19044.107272,-1498.081360,-1531.867501
3245,2022-10-09,19446.426194,-1461.846350,-1517.863271
3246,2022-10-16,19268.092801,-1431.023905,-1500.495398
3247,2022-10-23,19567.007398,-1366.722262,-1473.740771


In [12]:
df["Lag_1"] = (
    df["Close"].shift(1)
)

df["Lag_2"] = (
    df["Close"].shift(2)
)

df["Lag_3"] = (
    df["Close"].shift(3)
)

display(
    df[
        [
            "Date",
            "Close",
            "Lag_1",
            "Lag_2",
            "Lag_3"
        ]
    ].head(10)
)

,Date,Close,Lag_1,Lag_2,Lag_3
0,2013-05-05,115.910004,NaN,NaN,NaN
1,2013-05-06,112.300003,115.910004,NaN,NaN
2,2013-05-07,111.500000,112.300003,115.910004,NaN
3,2013-05-08,113.566002,111.500000,112.300003,115.910004
4,2013-05-09,112.669998,113.566002,111.500000,112.300003
5,2013-05-10,117.199997,112.669998,113.566002,111.500000
6,2013-05-11,115.242996,117.199997,112.669998,113.566002
7,2013-05-12,115.000000,115.242996,117.199997,112.669998
8,2013-05-13,117.980003,115.000000,115.242996,117.199997
9,2013-05-14,111.500000,117.980003,115.000000,115.242996


In [13]:
df["Volume_Change"] = (
    df["Volume"].pct_change()
)

display(
    df[
        [
            "Date",
            "Volume",
            "Volume_Change"
        ]
    ].head(10)
)

,Date,Volume,Volume_Change
0,2013-05-05,0.0,NaN
1,2013-05-06,0.0,NaN
2,2013-05-07,0.0,NaN
3,2013-05-08,0.0,NaN
4,2013-05-09,0.0,NaN
5,2013-05-10,0.0,NaN
6,2013-05-11,0.0,NaN
7,2013-05-12,0.0,NaN
8,2013-05-13,0.0,NaN
9,2013-05-14,0.0,NaN


In [14]:
df["Target"] = (
    df["Close"].shift(-1)
)

display(
    df[
        [
            "Date",
            "Close",
            "Target"
        ]
    ].head(10)
)

,Date,Close,Target
0,2013-05-05,115.910004,112.300003
1,2013-05-06,112.300003,111.500000
2,2013-05-07,111.500000,113.566002
3,2013-05-08,113.566002,112.669998
4,2013-05-09,112.669998,117.199997
5,2013-05-10,117.199997,115.242996
6,2013-05-11,115.242996,115.000000
7,2013-05-12,115.000000,117.980003
8,2013-05-13,117.980003,111.500000
9,2013-05-14,111.500000,114.220001


In [15]:
print("All columns:")
print()

for i, column in enumerate(df.columns, start=1):
    print(
        i,
        ".",
        column
    )

All columns:

1 . Date
2 . Open
3 . High
4 . Low
5 . Close
6 . Volume
7 . Return
8 . SMA_7
9 . SMA_14
10 . Volatility_7
11 . RSI_14
12 . MACD
13 . MACD_Signal
14 . Lag_1
15 . Lag_2
16 . Lag_3
17 . Volume_Change
18 . Target


In [16]:
print(
    "Missing values after feature engineering:"
)

display(
    df.isnull().sum()
)

Missing values after feature engineering:


Date               0
Open               0
High               0
Low                0
Close              0
Volume             0
Return             1
SMA_7              6
SMA_14            13
Volatility_7       7
RSI_14            13
MACD              25
MACD_Signal       33
Lag_1              1
Lag_2              2
Lag_3              3
Volume_Change    236
Target             1
dtype: int64

In [17]:
print(
    "Rows before removing NaN values:",
    len(df)
)

df = df.dropna().reset_index(
    drop=True
)

print(
    "Rows after removing NaN values:",
    len(df)
)

print(
    "\nRemaining missing values:"
)

print(
    df.isnull().sum().sum()
)

Rows before removing NaN values: 3248
Rows after removing NaN values: 3011

Remaining missing values:
0


In [18]:
target_check = np.allclose(
    df["Target"].iloc[:-1].values,
    df["Close"].iloc[1:].values
)

print(
    "Target equals next day's Close:",
    target_check
)

Target equals next day's Close: True


In [19]:
lag1_check = np.allclose(
    df["Lag_1"].iloc[1:].values,
    df["Close"].iloc[:-1].values
)

print(
    "Lag_1 correctly represents previous day's Close:",
    lag1_check
)

Lag_1 correctly represents previous day's Close: True


In [20]:
print("=" * 60)
print("DATA LEAKAGE CHECK")
print("=" * 60)

print("""
Target:
    Next-day Close

Features:
    Current-day and historical information only

Future information:
    Not intentionally included

Random shuffling:
    NOT used

Scaling:
    Will be fitted only on training data in the LSTM notebook
""")

DATA LEAKAGE CHECK

Target:
    Next-day Close

Features:
    Current-day and historical information only

Future information:
    Not intentionally included

Random shuffling:
    NOT used

Scaling:
    Will be fitted only on training data in the LSTM notebook



In [21]:
display(
    df.head(10)
)

,Date,Open,High,Low,Close,Volume,Return,SMA_7,SMA_14,Volatility_7,RSI_14,MACD,MACD_Signal,Lag_1,Lag_2,Lag_3,Volume_Change,Target
0,2013-12-27,763.280029,777.510010,713.599976,735.070007,46862700.0,-0.035316,677.298567,694.134282,0.059191,49.915908,-25.829491,-20.818665,761.979980,682.210022,665.580017,inf,727.830017
1,2013-12-28,737.979980,747.059998,705.349976,727.830017,32505800.0,-0.009849,694.751430,683.793570,0.056262,49.465334,-22.581305,-21.171193,735.070007,761.979980,682.210022,-0.306361,745.049988
2,2013-12-29,728.049988,748.609985,714.440002,745.049988,19011300.0,0.023659,713.018572,674.431427,0.056167,50.607340,-18.405421,-20.618038,727.830017,735.070007,761.979980,-0.415141,756.130005
3,2013-12-30,741.349976,766.599976,740.239990,756.130005,20707700.0,0.014872,724.835720,678.014287,0.048943,51.368868,-14.040092,-19.302449,745.049988,727.830017,735.070007,0.089231,754.010010
4,2013-12-31,760.320007,760.580017,738.169983,754.010010,20897300.0,-0.002804,737.468576,683.149288,0.048171,51.206191,-10.629078,-17.567775,756.130005,745.049988,727.830017,0.009156,771.400024
5,2014-01-01,754.969971,775.349976,754.969971,771.400024,22489400.0,0.023063,750.210005,700.913574,0.048136,52.534065,-6.448264,-15.343873,754.010010,756.130005,745.049988,0.076187,802.390015
6,2014-01-02,773.440002,820.309998,767.210022,802.390015,38489500.0,0.040174,755.982867,708.801431,0.025382,54.890040,-0.627078,-12.400514,771.400024,754.010010,756.130005,0.711451,818.719971
7,2014-01-03,802.849976,834.150024,789.119995,818.719971,37810100.0,0.020352,767.932861,722.615714,0.017000,56.125834,5.243502,-8.871711,802.390015,771.400024,754.010010,-0.017652,859.510010
8,2014-01-04,823.270020,859.510010,801.669983,859.510010,38005000.0,0.049822,786.744289,740.747859,0.017050,59.137146,13.037110,-4.489947,818.719971,802.390015,771.400024,0.005155,933.530029
9,2014-01-05,858.549988,952.400024,854.520020,933.530029,72898496.0,0.086119,813.670009,763.344291,0.028940,63.969876,24.899373,1.387917,859.510010,818.719971,802.390015,0.918129,953.289978


In [22]:
print(
    "Dataset shape:",
    df.shape
)

Dataset shape: (3011, 18)


In [23]:
# -----------------------------
# LSTM FEATURES
# -----------------------------

LSTM_FEATURES = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return",
    "Volatility_7"
]


# -----------------------------
# LIGHTGBM FEATURES
# -----------------------------

LIGHTGBM_FEATURES = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return",
    "SMA_7",
    "SMA_14",
    "Volatility_7",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Volume_Change"
]

TARGET = "Target"


print("LSTM features:")
print(LSTM_FEATURES)

print("\nLightGBM features:")
print(LIGHTGBM_FEATURES)

print("\nTarget:")
print(TARGET)

LSTM features:
['Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'Volatility_7']

LightGBM features:
['Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'SMA_7', 'SMA_14', 'Volatility_7', 'RSI_14', 'MACD', 'MACD_Signal', 'Lag_1', 'Lag_2', 'Lag_3', 'Volume_Change']

Target:
Target


In [24]:
missing_lstm = [
    feature
    for feature in LSTM_FEATURES
    if feature not in df.columns
]

missing_lgb = [
    feature
    for feature in LIGHTGBM_FEATURES
    if feature not in df.columns
]

print(
    "Missing LSTM features:",
    missing_lstm
)

print(
    "Missing LightGBM features:",
    missing_lgb
)

Missing LSTM features: []
Missing LightGBM features: []


In [25]:
n = len(df)

train_end = int(
    n * 0.70
)

validation_end = int(
    n * 0.85
)

train_df = df.iloc[
    :train_end
].copy()

validation_df = df.iloc[
    train_end:validation_end
].copy()

test_df = df.iloc[
    validation_end:
].copy()

print(
    "Total samples:",
    len(df)
)

print(
    "\nTraining samples:",
    len(train_df)
)

print(
    "Validation samples:",
    len(validation_df)
)

print(
    "Test samples:",
    len(test_df)
)

Total samples: 3011

Training samples: 2107
Validation samples: 452
Test samples: 452


In [26]:
print(
    "Train percentage:",
    round(
        len(train_df) / len(df) * 100,
        2
    ),
    "%"
)

print(
    "Validation percentage:",
    round(
        len(validation_df) / len(df) * 100,
        2
    ),
    "%"
)

print(
    "Test percentage:",
    round(
        len(test_df) / len(df) * 100,
        2
    ),
    "%"
)

Train percentage: 69.98 %
Validation percentage: 15.01 %
Test percentage: 15.01 %


In [27]:
print("=" * 60)
print("CHRONOLOGICAL DATA SPLIT")
print("=" * 60)

print("\nTRAIN")
print(
    train_df["Date"].min(),
    "→",
    train_df["Date"].max()
)

print("\nVALIDATION")
print(
    validation_df["Date"].min(),
    "→",
    validation_df["Date"].max()
)

print("\nTEST")
print(
    test_df["Date"].min(),
    "→",
    test_df["Date"].max()
)

CHRONOLOGICAL DATA SPLIT

TRAIN
2013-12-27 00:00:00 → 2019-10-03 00:00:00

VALIDATION
2019-10-04 00:00:00 → 2021-01-07 00:00:00

TEST
2021-01-08 00:00:00 → 2022-10-16 00:00:00


In [28]:
assert (
    train_df["Date"].max()
    <
    validation_df["Date"].min()
)

assert (
    validation_df["Date"].max()
    <
    test_df["Date"].min()
)

print(
    "✓ Chronological split verified successfully."
)

✓ Chronological split verified successfully.


In [29]:
train_dates = set(
    train_df["Date"]
)

validation_dates = set(
    validation_df["Date"]
)

test_dates = set(
    test_df["Date"]
)

print(
    "Train ∩ Validation:",
    len(
        train_dates.intersection(
            validation_dates
        )
    )
)

print(
    "Validation ∩ Test:",
    len(
        validation_dates.intersection(
            test_dates
        )
    )
)

print(
    "Train ∩ Test:",
    len(
        train_dates.intersection(
            test_dates
        )
    )
)

Train ∩ Validation: 0
Validation ∩ Test: 0
Train ∩ Test: 0


In [30]:
train_df.to_csv(
    OUTPUT_PATH / "train.csv",
    index=False
)

validation_df.to_csv(
    OUTPUT_PATH / "validation.csv",
    index=False
)

test_df.to_csv(
    OUTPUT_PATH / "test.csv",
    index=False
)

print(
    "Train, validation and test datasets saved."
)

Train, validation and test datasets saved.


In [31]:
df.to_csv(
    OUTPUT_PATH / "engineered_data.csv",
    index=False
)

print(
    "Complete engineered dataset saved."
)

print(
    "Location:",
    (
        OUTPUT_PATH /
        "engineered_data.csv"
    ).resolve()
)

Complete engineered dataset saved.
Location: C:\Crypto_Forecasting_Project\LSTM_Light_BGM\data\processed\engineered_data.csv


In [32]:
split_info = pd.DataFrame({

    "Split": [
        "Train",
        "Validation",
        "Test"
    ],

    "Rows": [
        len(train_df),
        len(validation_df),
        len(test_df)
    ],

    "Percentage": [
        len(train_df) / len(df) * 100,
        len(validation_df) / len(df) * 100,
        len(test_df) / len(df) * 100
    ],

    "Start_Date": [
        train_df["Date"].min(),
        validation_df["Date"].min(),
        test_df["Date"].min()
    ],

    "End_Date": [
        train_df["Date"].max(),
        validation_df["Date"].max(),
        test_df["Date"].max()
    ]
})

display(
    split_info
)

split_info.to_csv(
    OUTPUT_PATH / "split_info.csv",
    index=False
)

,Split,Rows,Percentage,Start_Date,End_Date
0,Train,2107,69.976752,2013-12-27,2019-10-03
1,Validation,452,15.011624,2019-10-04,2021-01-07
2,Test,452,15.011624,2021-01-08,2022-10-16


In [33]:
print("=" * 65)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 65)

print("\nTotal rows:", len(df))

print(
    "Number of features:",
    len(LIGHTGBM_FEATURES)
)

print(
    "Target:",
    TARGET
)

print(
    "\nTrain rows:",
    len(train_df)
)

print(
    "Validation rows:",
    len(validation_df)
)

print(
    "Test rows:",
    len(test_df)
)

print(
    "\nMissing values:",
    df.isnull().sum().sum()
)

print(
    "Duplicate dates:",
    df["Date"].duplicated().sum()
)

print(
    "Chronological:",
    df["Date"].is_monotonic_increasing
)

FINAL PREPROCESSING SUMMARY

Total rows: 3011
Number of features: 16
Target: Target

Train rows: 2107
Validation rows: 452
Test rows: 452

Missing values: 0
Duplicate dates: 0
Chronological: True


In [34]:
feature_summary = pd.DataFrame({

    "Feature": LIGHTGBM_FEATURES,

    "Used_by_LSTM": [
        feature in LSTM_FEATURES
        for feature in LIGHTGBM_FEATURES
    ],

    "Used_by_LightGBM": [
        True
        for feature in LIGHTGBM_FEATURES
    ]
})

display(
    feature_summary
)

,Feature,Used_by_LSTM,Used_by_LightGBM
0,Open,True,True
1,High,True,True
2,Low,True,True
3,Close,True,True
4,Volume,True,True
5,Return,True,True
6,SMA_7,False,True
7,SMA_14,False,True
8,Volatility_7,True,True
9,RSI_14,False,True


In [35]:
print("LightGBM features:")
print(LIGHTGBM_FEATURES)

print("\nNumber of LightGBM features:")
print(len(LIGHTGBM_FEATURES))

print("\nLSTM features:")
print(LSTM_FEATURES)

print("\nNumber of LSTM features:")
print(len(LSTM_FEATURES))

LightGBM features:
['Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'SMA_7', 'SMA_14', 'Volatility_7', 'RSI_14', 'MACD', 'MACD_Signal', 'Lag_1', 'Lag_2', 'Lag_3', 'Volume_Change']

Number of LightGBM features:
16

LSTM features:
['Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'Volatility_7']

Number of LSTM features:
7


# Preprocessing Completed

The Bitcoin dataset has been transformed into a
machine-learning-ready time-series dataset.

## Completed

- Bitcoin OHLCV data loaded
- Daily returns calculated
- SMA-7 calculated
- SMA-14 calculated
- 7-day volatility calculated
- RSI-14 calculated
- MACD calculated
- MACD signal calculated
- Lag-1, Lag-2 and Lag-3 created
- Volume change calculated
- Next-day Close target created
- NaN values generated by rolling/lag operations removed
- Target alignment verified
- Data leakage checks performed
- Chronological 70/15/15 split created

## Saved Files

- `bitcoin_clean.csv`
- `engineered_data.csv`
- `train.csv`
- `validation.csv`
- `test.csv`
- `split_info.csv`

## Next Notebook

### `03_lstm_model.ipynb`

The next notebook will implement the LSTM model using
a 30-day lookback sequence and leakage-safe scaling.